# Bài 3
Đây là notebook chứa mã nguồn đầy đủ của bài 3.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai03(data_dir=None, w_growth=0.15, w_productivity=0.15, w_spillover=0.20,
                w_export=0.15, w_employment=0.10, w_ai=0.20, w_risk=0.15):
    sectors = ['Nông nghiệp', 'Chế biến', 'Xây dựng', 'Khai khoáng', 'Bán lẻ',
               'Tài chính', 'Logistics', 'CNTT', 'Giáo dục', 'Y tế']
    # [growth%, productivity, spillover, export, labor(M), ai_readiness, automation_risk]
    X = np.array([
        [3.27,  103,  0.35,  40.5, 13.2, 15, 18],
        [9.64,  241,  0.78, 290.9, 11.5, 55, 42],
        [7.45,  169,  0.42,   2.5,  4.8, 20, 25],
        [-1.2, 1290,  0.30,   8.2,  0.3, 30, 55],
        [7.10,  145,  0.55,   5.5,  7.8, 48, 38],
        [7.36, 1072,  0.85,   1.2, 0.55, 72, 52],
        [9.93,  321,  0.72,   3.1, 1.95, 42, 35],
        [7.85,  714,  0.92,  178,  0.62, 88, 28],
        [6.42,  206,  0.65,   0,   2.15, 38, 22],
        [6.85,  437,  0.60,   0,   0.75, 45, 18],
    ], float)

    good = X[:, :6]
    bad  = X[:, 6]

    # Min-max normalization
    Gn = (good - good.min(0)) / (np.ptp(good, axis=0) + 1e-9)
    # Risk is bad, so inverse normalization: (max - x) / (max - min)
    Rn = (bad.max() - bad) / (np.ptp(bad) + 1e-9)

    # Combine all normalized columns
    norm_matrix = np.column_stack((Gn, Rn))
    col_names = ['Tăng trưởng', 'Năng suất', 'Lan tỏa', 'Xuất khẩu', 'Việc làm', 'AI Ready', 'Rủi ro (đảo dấu)']

    # Base weights normalized to sum=1
    w = np.array([w_growth, w_productivity, w_spillover, w_export, w_employment, w_ai, w_risk], float)
    w_sum = w.sum() if w.sum() > 0 else 1
    w = w / w_sum

    score = norm_matrix @ w
    idx = np.argsort(-score)

    ranking = []
    for i in idx:
        ranking.append({
            'sector_name_vi': sectors[i],
            'Priority': round(float(score[i]), 4),
            'rank': int(np.where(idx == i)[0][0]) + 1,
        })

    # Sensitivity a6 (AI Readiness)
    a6_values = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    heatmap_data = []
    sens_top3 = {}
    for a6 in a6_values:
        w_temp = np.array([0.15, 0.15, 0.20, 0.15, 0.10, a6, 0.15])
        w_temp = w_temp / w_temp.sum()
        s_temp = norm_matrix @ w_temp
        heatmap_data.append(s_temp.tolist())
        top3_idx = np.argsort(-s_temp)[:3]
        sens_top3[str(a6)] = [sectors[i] for i in top3_idx]
    heatmap_data = np.array(heatmap_data).T.tolist()

    # Scenarios comparison
    w_growth_orient = np.array([0.30, 0.20, 0.15, 0.15, 0.05, 0.10, 0.05])
    w_inclusive = np.array([0.10, 0.05, 0.25, 0.05, 0.30, 0.10, 0.15])
    w_growth_orient = w_growth_orient / w_growth_orient.sum()
    w_inclusive = w_inclusive / w_inclusive.sum()

    score_g = norm_matrix @ w_growth_orient
    score_i = norm_matrix @ w_inclusive

    idx_g = np.argsort(-score_g)
    idx_i = np.argsort(-score_i)

    scenario_comparison = {
        'growth': {
            'top3': [sectors[i] for i in idx_g[:3]],
            'scores': {sectors[i]: round(float(score_g[i]), 4) for i in idx_g}
        },
        'inclusive': {
            'top3': [sectors[i] for i in idx_i[:3]],
            'scores': {sectors[i]: round(float(score_i[i]), 4) for i in idx_i}
        }
    }

    return {
        'sectors': sectors,
        'col_names': col_names,
        'norm_matrix': norm_matrix.tolist(),
        'ranking': ranking,
        'a6_values': a6_values,
        'heatmap_data': heatmap_data,
        'sensitivity_top3': sens_top3,
        'scenario_comparison': scenario_comparison
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai03()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())